# v10 — **Deep validation and reopened pillars**
Continues from HEAD `81057b2` (v9 revalidation). Nothing from v1–v9 is overwritten.

## Purpose
v9's conclusions are treated as **hypotheses to attack**, not findings to defend. Three jobs:
1. Stress the two provisional candidates (Macro FCI, Micro EPS) hard enough to break them if they are fragile.
2. **Reopen Valuation and Technical properly** — v9 rejected both after testing few constructions.
3. **Correct v9's own methodology** where it was wrong. One correction below changes how every previous robustness claim should be read.

## Research log
Every test in this notebook appends to a log with hypothesis, construction, sample, result and conclusion. Exported to `output/v10/research_log.csv`.

In [1]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt, warnings, json
from pathlib import Path
import statsmodels.api as sm
from scipy.stats import spearmanr, rankdata
from IPython.display import display, Markdown
warnings.filterwarnings('ignore')
pd.options.display.float_format = '{:,.3f}'.format; pd.options.display.width = 240
plt.rcParams.update({'figure.dpi': 110, 'axes.spines.top': False, 'axes.spines.right': False, 'axes.grid': True, 'grid.alpha': .25, 'font.size': 10})
ROOT = Path('..').resolve(); OUT = ROOT/'output'/'v10'; CH = OUT/'charts'
for p in (OUT, CH): p.mkdir(parents=True, exist_ok=True)
COUNTRIES = ['US', 'Eurozone', 'UK', 'Switzerland', 'Japan']
SEED = 7; NBOOT = 2000; NPERM = 2000
LOG = []
def log(test_id, pillar, hypothesis, construction, sample, result, conclusion):
    LOG.append(dict(id=test_id, pillar=pillar, hypothesis=hypothesis, construction=construction,
                    sample=sample, result=result, conclusion=conclusion))

SPEC = {
 'US':          dict(file='US_raw_data.xlsx',  eps='Monthly_EPS', fci=('GSFCI Index', -1),   y2='USGGT02Y Index', oas='LF98OAS Index',  fx=None),
 'Eurozone':    dict(file='EUR_raw_data.xlsx', eps='BEst EPS',    fci=('BFCIEU Index', +1),  y2='GDBR2 Index',    oas='LECPOAS Index', fx=('EURUSD BGN Curncy', 'direct')),
 'UK':          dict(file='UK_raw_data.xlsx',  eps='BEst EPS',    fci=('GSGBFCI Index', -1), y2='GUKG2 Index',    oas='LC61OAS Index', fx=('GBPUSD BGN Curncy', 'direct')),
 'Switzerland': dict(file='CH_raw_data.xlsx',  eps='BEst EPS',    fci=('GSEAFCI Index', -1), y2='GSWISS02 Index', oas='LECPOAS Index', fx=('USDCHF Curncy', 'invert')),
 'Japan':       dict(file='JPY_raw_data.xlsx', eps='BEst EPS',    fci=('GSJPFCI Index', -1), y2='GJGB2 Index',    oas='LJC1OAS Index', fx=('USDJPY BGN Curncy', 'invert')),
}
def monthly(df):
    d = df.dropna(subset=['Date']).copy(); d['month'] = pd.to_datetime(d['Date']).dt.to_period('M')
    return d.sort_values('Date').drop_duplicates('month', keep='last').set_index('month')
def load(s):
    d = monthly(pd.read_excel(ROOT/'data'/'raw'/s['file']))
    o = pd.DataFrame({'pe': d['BEst P/E Ratio'], 'eps': d[s['eps']], 'y2': d[s['y2']], 'oas': d[s['oas']]})
    o['fci'] = s['fci'][1] * d[s['fci'][0]]
    o['fx'] = 1.0 if s['fx'] is None else (d[s['fx'][0]] if s['fx'][1] == 'direct' else 1.0 / d[s['fx'][0]])
    return o
fr = {c: load(SPEC[c]) for c in COUNTRIES}
IDX = pd.period_range(max(fr[c].pe.first_valid_index() for c in COUNTRIES), min(fr[c].index.max() for c in COUNTRIES), freq='M')
PE, EPS, Y2, OAS, FCI, FX = [pd.DataFrame({c: fr[c][k] for c in COUNTRIES}).reindex(IDX) for k in ('pe', 'eps', 'y2', 'oas', 'fci', 'fx')]
VIX = monthly(pd.read_excel(ROOT/'data'/'raw'/'US_raw_data.xlsx'))['VIX Index'].reindex(IDX)
AS_OF = IDX[-1]; PX = PE * EPS; PX_USD = PX * FX; R_USD = PX_USD.pct_change()
cs_z = lambda d: d.sub(d.mean(axis=1), axis=0).div(d.std(axis=1, ddof=1), axis=0).where(d.notna().all(axis=1))
z_own = lambda x, w=60: (x - x.expanding(w).mean()) / x.expanding(w).std()
own_sd = lambda d, w=36: d / d.expanding(w).std()
TGT = {}
for h in (1, 3, 6, 12):
    tot = PX_USD.shift(-h) / PX_USD - 1
    TGT[('total', h)] = tot.sub(tot.mean(axis=1), axis=0)
for h in (3, 6):
    for nm, ser in (('multiple', np.log(PE.shift(-h) / PE)), ('earnings', np.log(EPS.shift(-h) / EPS))):
        TGT[(nm, h)] = ser.sub(ser.mean(axis=1), axis=0)

def ic_series(sig, tgt, sample=None):
    o = {}
    for t in sig.index:
        if sample is not None and t not in sample: continue
        a, b = sig.loc[t], tgt.loc[t]
        if a.notna().all() and b.notna().all() and a.nunique() > 1: o[t] = spearmanr(a, b).statistic
    return pd.Series(o).sort_index()
def win(s, w): return s[(s.index >= w[0]) & (s.index <= w[1])]
def hac_t(x, lags):
    x = np.asarray(x, float); x = x[~np.isnan(x)]
    if len(x) < 10: return np.nan, np.nan
    m = sm.OLS(x, np.ones(len(x))).fit(cov_type='HAC', cov_kwds={'maxlags': lags})
    return float(m.tvalues[0]), float(m.pvalues[0])
def mbb(x, L, n=NBOOT, seed=SEED):
    x = np.asarray(x, float); x = x[~np.isnan(x)]; T = len(x)
    if T < 12: return (np.nan, np.nan, np.nan)
    rng = np.random.default_rng(seed); nb = int(np.ceil(T / L)); out = np.empty(n)
    for i in range(n):
        s = rng.integers(0, max(1, T - L + 1), nb)
        out[i] = np.concatenate([x[a:a + L] for a in s])[:T].mean()
    lo, hi = np.percentile(out, [2.5, 97.5])
    p = (1 + int(((out - out.mean()) >= x.mean()).sum())) / (1 + n)
    return float(lo), float(hi), float(p)

## 1. METHOD CORRECTION — the six non-overlapping offsets were never six independent tests
v8 and v9 both used "k of 6 offsets positive" as a robustness criterion. That criterion is **not valid as stated**, and this changes how earlier robustness claims should be read.

The six offsets *partition* one monthly IC series. Their mean is therefore (up to an end-effect) the mean of the whole series — they cannot be six independent confirmations of anything.
They are a **sensitivity check on the start month**, which is useful, but not evidence of replication.

In [2]:
EPSREV6 = EPS / EPS.shift(6) - 1
FCI6 = own_sd(FCI - FCI.shift(6))
Z0 = {'MACRO_fci6': cs_z(FCI6), 'MICRO_eps6': cs_z(EPSREV6)}
valid0 = pd.Series({t: all(Z0[k].loc[t].notna().all() for k in Z0) for t in IDX})
demo = ic_series(Z0['MACRO_fci6'], TGT[('total', 6)])
offs = [demo.iloc[o::6].mean() for o in range(6)]
print(f'Mean of the six offset means : {np.mean(offs):+.4f}')
print(f'Mean of the full IC series   : {demo.mean():+.4f}   <- same number (end-effects aside)')
print(f'Offset means: {[f"{v:+.3f}" for v in offs]}  -> spread {max(offs)-min(offs):.3f} measures START-DATE SENSITIVITY, not replication.')
print('\nCONSEQUENCE: "5/6 offsets positive" in v8/v9 overstated the evidence. It is retained below only as a dispersion diagnostic.')
print('Replaced by: moving-block bootstrap on the FULL series with block length >= 2x horizon, and HAC with lags = horizon.')
rows = []
for L in (3, 6, 12, 18):
    lo, hi, p = mbb(demo.values, L)
    rows.append({'block length': L, 'boot 2.5%': lo, 'boot 97.5%': hi, 'CI width': hi - lo, 'add-one p': p})
BLK = pd.DataFrame(rows).set_index('block length')
t6, p6 = hac_t(demo.values, 6); t3, _ = hac_t(demo.values, 3); t12, _ = hac_t(demo.values, 12)
display(BLK.style.format({'boot 2.5%': '{:+.3f}', 'boot 97.5%': '{:+.3f}', 'CI width': '{:.3f}', 'add-one p': '{:.3f}'}))
print(f'HAC t on the same 6M series: lags=3 -> {t3:+.2f}, lags=6 -> {t6:+.2f}, lags=12 -> {t12:+.2f}')
print('Longer blocks and longer HAC lags widen the interval, as they should with 6M overlap. v10 uses L=2h and lags=h.')
log('M01', 'method', 'The six offsets are six independent robustness tests',
    'compare mean-of-offsets with full-series mean; vary bootstrap block length',
    'full common sample, 6M horizon',
    f'mean-of-offsets {np.mean(offs):+.4f} = full-series mean {demo.mean():+.4f}; CI width grows {BLK.loc[3,"CI width"]:.3f}->{BLK.loc[18,"CI width"]:.3f} as block length 3->18',
    'REFUTED. Offsets are a start-date sensitivity diagnostic, not replication. v8/v9 robustness claims overstated; superseded by block bootstrap with L=2h.')

Mean of the six offset means : +0.1742
Mean of the full IC series   : +0.1742   <- same number (end-effects aside)
Offset means: ['+0.000', '+0.185', '+0.164', '+0.209', '+0.224', '+0.264']  -> spread 0.264 measures START-DATE SENSITIVITY, not replication.

CONSEQUENCE: "5/6 offsets positive" in v8/v9 overstated the evidence. It is retained below only as a dispersion diagnostic.
Replaced by: moving-block bootstrap on the FULL series with block length >= 2x horizon, and HAC with lags = horizon.


,boot 2.5%,boot 97.5%,CI width,add-one p
block length,,,,
3,+0.074,+0.266,0.191,0.001
6,+0.061,+0.277,0.216,0.001
12,+0.058,+0.299,0.242,0.002
18,+0.058,+0.299,0.241,0.000


HAC t on the same 6M series: lags=3 -> +3.32, lags=6 -> +2.96, lags=12 -> +2.78
Longer blocks and longer HAC lags widen the interval, as they should with 6M overlap. v10 uses L=2h and lags=h.


## 2. Signal library for v10
Macro and Micro variants for the robustness sweep; Valuation and Technical reopened with economically distinct constructions.
**Technical includes a new family: path-based signals.** Momentum ratios can be rewritten as (ΔP/E + ΔEPS), so they inherit earnings information by construction.
Volatility, drawdown and trend-quality depend on the *shape of the price path*, which no EPS series can reproduce — they are orthogonal to earnings by construction, not by residualisation.

In [3]:
RLOC = PX.pct_change(); RREL = R_USD.sub(R_USD.mean(axis=1), axis=0)
EPSG12 = EPS / EPS.shift(12) - 1; ERP = 100 / PE - Y2
def trend_r2(df, w=12):
    out = pd.DataFrame(index=df.index, columns=df.columns, dtype=float)
    x = np.arange(w); xc = x - x.mean()
    for c in df.columns:
        lp = np.log(df[c])
        for i in range(w - 1, len(df)):
            y = lp.iloc[i - w + 1:i + 1].values
            if np.isnan(y).any(): continue
            yc = y - y.mean(); den = (yc**2).sum()
            if den <= 0: continue
            b = (xc * yc).sum() / (xc**2).sum()
            out.iloc[i, df.columns.get_loc(c)] = 1 - ((yc - b * xc)**2).sum() / den
    return out
def max_dd(df, w=12):
    return df.rolling(w).apply(lambda s: (s / np.maximum.accumulate(s) - 1).min(), raw=True)

MACRO_V = {
 'fci chg 1M':  own_sd(FCI - FCI.shift(1)),  'fci chg 3M': own_sd(FCI - FCI.shift(3)),
 'fci chg 6M':  own_sd(FCI - FCI.shift(6)),  'fci chg 12M': own_sd(FCI - FCI.shift(12)),
 'fci level z': z_own(FCI),                  'fci chg 6M (rank)': (FCI - FCI.shift(6)).rank(axis=1),
 'fci accel (3M vs prior 3M)': own_sd((FCI - FCI.shift(3)) - (FCI.shift(3) - FCI.shift(6))),
 'credit chg 6M (tighten+)': -own_sd(OAS - OAS.shift(6)),
 'fci + credit blend': cs_z(own_sd(FCI - FCI.shift(6))) + cs_z(-own_sd(OAS - OAS.shift(6))),
}
MICRO_V = {
 'eps rev 1M': EPS / EPS.shift(1) - 1,   'eps rev 3M': EPS / EPS.shift(3) - 1,
 'eps rev 6M': EPS / EPS.shift(6) - 1,   'eps rev 12M': EPS / EPS.shift(12) - 1,
 'eps rev 6M (log)': np.log(EPS / EPS.shift(6)),
 'eps rev 6M (own-sd scaled)': own_sd(EPS / EPS.shift(6) - 1),
 'eps rev 6M lagged 1M': (EPS / EPS.shift(6) - 1).shift(1),
 'eps rev 6M lagged 2M': (EPS / EPS.shift(6) - 1).shift(2),
 'eps rev 3M minus 12M (accel)': (EPS / EPS.shift(3) - 1) - (EPS / EPS.shift(12) - 1) / 4,
 'eps stability 12M (inv vol)': -EPS.pct_change().rolling(12).std(),
}
VAL_V = {
 'P/E z own history (cheap+)':  -z_own(PE),
 'P/E vs peers (cheap+)':       -z_own(PE.div(PE.mean(axis=1), axis=0)),
 'earnings yield z':             z_own(100 / PE),
 'ERP z (E/P - 2Y)':             z_own(ERP),
 'ERP vs credit (E/P-2Y-OAS)':   z_own(ERP - OAS),
 'P/E chg 6M reversal':         -own_sd(np.log(PE / PE.shift(6))),
 'P/E chg 12M reversal':        -own_sd(np.log(PE / PE.shift(12))),
 'P/E chg 3M reversal':         -own_sd(np.log(PE / PE.shift(3))),
 'P/E percentile own 10y':      -PE.rolling(120, min_periods=60).rank(pct=True),
 'cheap AND upgrading (gate)':  (-z_own(PE)).where(EPSREV6 > 0, 0.0),
 'cheap AND easing FCI (gate)': (-z_own(PE)).where(FCI - FCI.shift(6) > 0, 0.0),
 'cheap AND low vol (gate)':    (-z_own(PE)).where(RLOC.rolling(12).std().rank(axis=1) <= 2, 0.0),
}
TECH_V = {
 'mom 1M (reversal, inv)':  -(PX / PX.shift(1) - 1),
 'mom 3M':                    PX / PX.shift(3) - 1,
 'mom 6M':                    PX / PX.shift(6) - 1,
 'mom 12M':                   PX / PX.shift(12) - 1,
 'mom 12-1M':                 PX.shift(1) / PX.shift(13) - 1,
 'mom 6-1M':                  PX.shift(1) / PX.shift(7) - 1,
 'relative strength 6M':      (PX / PX.shift(6)).div((PX / PX.shift(6)).mean(axis=1), axis=0),
 'price vs 6M MA':            PX / PX.rolling(6).mean() - 1,
 'vol-adjusted mom 12-1M':   (PX.shift(1) / PX.shift(13) - 1) / RLOC.rolling(12).std(),
 'PATH low realised vol 12M': -RLOC.rolling(12).std(),
 'PATH small drawdown 12M':   max_dd(PX, 12),
 'PATH trend quality R2 12M':  trend_r2(PX, 12),
 'PATH downside semidev 12M': -np.sqrt((RLOC.clip(upper=0) ** 2).rolling(12).mean()),   # standard semideviation: positive months contribute 0, so always defined
}
FAMS = {'Macro': MACRO_V, 'Micro': MICRO_V, 'Valuation': VAL_V, 'Technical': TECH_V}
Z = {f'{p}|{k}': cs_z(v.reindex(IDX)) for p, fam in FAMS.items() for k, v in fam.items()}
cover = pd.DataFrame({k: Z[k].notna().all(axis=1) for k in Z})
firsts = {k: (cover[k][cover[k]].index.min() if cover[k].any() else None) for k in cover}
late = sorted([(str(v), k) for k, v in firsts.items() if v is not None], reverse=True)[:5]
print('Coverage diagnostic - signals that start latest (these set the common sample):')
for d_, k in late: print(f'   {d_}  {k}')
dead = [k for k, v in firsts.items() if v is None]
if dead: print('SIGNALS WITH NO VALID MONTHS (dropped):', dead)
for k in dead: Z.pop(k)
cover = cover.drop(columns=dead)
valid = cover.all(axis=1)
SAMPLE = valid[valid].index
assert len(SAMPLE) > 0, 'empty common sample'
DISC = (SAMPLE.min(), pd.Period('2016-12', 'M')); VALW = (pd.Period('2017-07', 'M'), SAMPLE.max())
print(f'{len(Z)} signals across 4 pillars. COMMON SAMPLE: {SAMPLE.min()} -> {SAMPLE.max()} ({len(SAMPLE)} months)')
print(f'  discovery {DISC[0]}..{DISC[1]}   embargo 2017-01..2017-06   validation {VALW[0]}..{VALW[1]}')
print('Every statistic below uses this one sample. Signals needing a 10y rolling window shorten it - noted where relevant.')

Coverage diagnostic - signals that start latest (these set the common sample):
   2010-11  Valuation|earnings yield z
   2010-11  Valuation|cheap AND upgrading (gate)
   2010-11  Valuation|cheap AND low vol (gate)
   2010-11  Valuation|cheap AND easing FCI (gate)
   2010-11  Valuation|P/E z own history (cheap+)
44 signals across 4 pillars. COMMON SAMPLE: 2010-11 -> 2026-04 (159 months)
  discovery 2010-11..2016-12   embargo 2017-01..2017-06   validation 2017-07..2026-04
Every statistic below uses this one sample. Signals needing a 10y rolling window shorten it - noted where relevant.


## 3. Standalone robustness sweep (all four pillars, identical sample)

In [4]:
def full_eval(name, sig):
    r = {'signal': name}
    for lab, w, h, lags, L in (('disc3', DISC, 3, 3, 6), ('val3', VALW, 3, 3, 6), ('full3', (SAMPLE.min(), SAMPLE.max()), 3, 3, 6),
                               ('val6', VALW, 6, 6, 12), ('full6', (SAMPLE.min(), SAMPLE.max()), 6, 6, 12)):
        ic = win(ic_series(sig, TGT[('total', h)], SAMPLE), w)
        t, _ = hac_t(ic.values, lags); lo, hi, p = mbb(ic.values, L)
        r[f'{lab} IC'] = ic.mean(); r[f'{lab} t'] = t
        if lab in ('val3', 'full3', 'full6'): r[f'{lab} lo'] = lo; r[f'{lab} p'] = p
    ic3v = win(ic_series(sig, TGT[('total', 3)], SAMPLE), VALW); hlf = len(ic3v) // 2
    r['val 1st half'] = ic3v.iloc[:hlf].mean(); r['val 2nd half'] = ic3v.iloc[hlf:].mean()
    r['leg multiple'] = win(ic_series(sig, TGT[('multiple', 3)], SAMPLE), VALW).mean()
    r['leg earnings'] = win(ic_series(sig, TGT[('earnings', 3)], SAMPLE), VALW).mean()
    return r
SWEEP = pd.DataFrame([full_eval(k, v) for k, v in Z.items()]).set_index('signal')
SWEEP['pillar'] = [k.split('|')[0] for k in SWEEP.index]
fmtS = {c: '{:+.3f}' for c in SWEEP.columns if 'IC' in c or 'leg' in c or 'half' in c or c.endswith(' lo')} | \
       {c: '{:+.2f}' for c in SWEEP.columns if c.endswith(' t')} | {c: '{:.3f}' for c in SWEEP.columns if c.endswith(' p')}
for p_ in FAMS:
    print(f'=== {p_} ==='); display(SWEEP[SWEEP.pillar == p_].drop(columns='pillar').style.format(fmtS))
SWEEP.to_csv(OUT/'standalone_sweep.csv')

=== Macro ===


,disc3 IC,disc3 t,val3 IC,val3 t,val3 lo,val3 p,full3 IC,full3 t,full3 lo,full3 p,val6 IC,val6 t,full6 IC,full6 t,full6 lo,full6 p,val 1st half,val 2nd half,leg multiple,leg earnings
signal,,,,,,,,,,,,,,,,,,,,
Macro|fci chg 1M,+0.079,+1.27,-0.020,-0.43,-0.118,0.671,+0.025,+0.67,-0.047,0.260,+0.090,+1.84,+0.081,+2.04,+0.025,0.006,-0.057,+0.016,-0.045,+0.101
Macro|fci chg 3M,+0.041,+0.45,+0.100,+1.54,-0.037,0.071,+0.072,+1.39,-0.028,0.086,+0.124,+2.15,+0.095,+1.92,+0.024,0.010,+0.039,+0.160,-0.096,+0.184
Macro|fci chg 6M,+0.048,+0.51,+0.175,+2.49,+0.029,0.006,+0.119,+2.14,-0.003,0.015,+0.202,+2.38,+0.160,+2.53,+0.059,0.004,+0.198,+0.153,-0.024,+0.197
Macro|fci chg 12M,+0.115,+1.09,+0.152,+2.09,-0.019,0.017,+0.122,+2.07,-0.002,0.020,+0.135,+1.65,+0.135,+1.98,+0.007,0.023,+0.234,+0.071,-0.033,+0.158
Macro|fci level z,+0.039,+0.36,+0.126,+1.80,-0.011,0.029,+0.076,+1.27,-0.047,0.114,+0.105,+1.32,+0.079,+1.09,-0.061,0.140,+0.123,+0.129,+0.015,+0.196
Macro|fci chg 6M (rank),+0.055,+0.61,+0.157,+2.62,+0.037,0.003,+0.114,+2.29,+0.008,0.011,+0.134,+1.82,+0.135,+2.29,+0.049,0.008,+0.183,+0.132,-0.040,+0.197
Macro|fci accel (3M vs prior 3M),-0.048,-0.73,-0.072,-1.22,-0.202,0.870,-0.067,-1.54,-0.151,0.935,-0.001,-0.02,-0.003,-0.08,-0.088,0.527,-0.161,+0.016,-0.093,+0.017
Macro|credit chg 6M (tighten+),-0.142,-1.49,-0.073,-1.24,-0.189,0.917,-0.122,-2.29,-0.221,0.990,-0.126,-1.96,-0.117,-1.98,-0.220,0.981,-0.071,-0.074,-0.122,-0.035
Macro|fci + credit blend,-0.075,-0.76,+0.084,+1.39,-0.049,0.066,+0.006,+0.12,-0.105,0.447,+0.035,+0.43,+0.029,+0.43,-0.076,0.351,+0.084,+0.084,-0.089,+0.125


=== Micro ===


,disc3 IC,disc3 t,val3 IC,val3 t,val3 lo,val3 p,full3 IC,full3 t,full3 lo,full3 p,val6 IC,val6 t,full6 IC,full6 t,full6 lo,full6 p,val 1st half,val 2nd half,leg multiple,leg earnings
signal,,,,,,,,,,,,,,,,,,,,
Micro|eps rev 1M,-0.048,-0.64,+0.062,+1.41,-0.035,0.083,+0.016,+0.41,-0.065,0.340,+0.136,+2.61,+0.090,+1.81,+0.004,0.022,+0.064,+0.060,-0.092,+0.281
Micro|eps rev 3M,+0.025,+0.23,+0.085,+1.20,-0.067,0.114,+0.059,+0.99,-0.065,0.164,+0.081,+0.91,+0.111,+1.54,-0.027,0.075,+0.082,+0.089,-0.139,+0.424
Micro|eps rev 6M,+0.133,+1.29,+0.102,+1.21,-0.082,0.101,+0.113,+1.76,-0.024,0.041,+0.058,+0.54,+0.130,+1.62,-0.024,0.052,+0.143,+0.062,-0.078,+0.402
Micro|eps rev 12M,+0.080,+0.72,+0.028,+0.34,-0.138,0.364,+0.048,+0.75,-0.071,0.229,+0.062,+0.73,+0.105,+1.45,-0.009,0.070,+0.098,-0.040,-0.034,+0.265
Micro|eps rev 6M (log),+0.133,+1.29,+0.102,+1.21,-0.082,0.101,+0.113,+1.76,-0.024,0.041,+0.058,+0.54,+0.130,+1.62,-0.024,0.052,+0.143,+0.062,-0.078,+0.402
Micro|eps rev 6M (own-sd scaled),+0.154,+1.71,+0.161,+2.29,+0.029,0.008,+0.159,+2.97,+0.052,0.002,+0.143,+1.55,+0.171,+2.46,+0.046,0.004,+0.234,+0.089,-0.021,+0.297
Micro|eps rev 6M lagged 1M,+0.139,+1.43,+0.108,+1.21,-0.078,0.104,+0.114,+1.77,-0.024,0.042,+0.034,+0.30,+0.112,+1.35,-0.042,0.097,+0.207,+0.011,-0.073,+0.382
Micro|eps rev 6M lagged 2M,+0.121,+1.25,+0.052,+0.69,-0.112,0.224,+0.075,+1.29,-0.044,0.103,+0.005,+0.05,+0.080,+1.03,-0.048,0.145,+0.127,-0.022,-0.098,+0.344
Micro|eps rev 3M minus 12M (accel),-0.048,-0.60,+0.064,+1.25,-0.039,0.086,+0.022,+0.51,-0.073,0.311,+0.041,+0.54,+0.027,+0.46,-0.106,0.325,+0.034,+0.093,-0.076,+0.204


=== Valuation ===


,disc3 IC,disc3 t,val3 IC,val3 t,val3 lo,val3 p,full3 IC,full3 t,full3 lo,full3 p,val6 IC,val6 t,full6 IC,full6 t,full6 lo,full6 p,val 1st half,val 2nd half,leg multiple,leg earnings
signal,,,,,,,,,,,,,,,,,,,,
Valuation|P/E z own history (cheap+),+0.110,+1.41,-0.090,-1.35,-0.252,0.919,-0.001,-0.01,-0.119,0.519,-0.127,-1.33,+0.024,+0.31,-0.179,0.390,-0.134,-0.047,+0.130,-0.173
Valuation|P/E vs peers (cheap+),+0.136,+1.24,-0.064,-0.90,-0.231,0.820,+0.027,+0.43,-0.106,0.345,-0.065,-0.68,+0.071,+0.78,-0.134,0.251,-0.080,-0.049,+0.138,-0.234
Valuation|earnings yield z,+0.041,+0.53,-0.072,-1.12,-0.220,0.887,-0.016,-0.32,-0.120,0.636,-0.112,-1.23,-0.016,-0.22,-0.194,0.568,-0.098,-0.047,+0.144,-0.192
Valuation|ERP z (E/P - 2Y),+0.072,+0.77,-0.029,-0.50,-0.140,0.702,+0.025,+0.48,-0.068,0.303,+0.012,+0.14,+0.122,+1.82,-0.029,0.051,+0.032,-0.089,+0.071,-0.153
Valuation|ERP vs credit (E/P-2Y-OAS),+0.123,+1.23,+0.052,+0.76,-0.085,0.238,+0.097,+1.69,-0.029,0.048,+0.108,+1.05,+0.179,+2.42,+0.033,0.012,+0.048,+0.056,+0.091,-0.055
Valuation|P/E chg 6M reversal,+0.057,+0.84,+0.070,+1.08,-0.074,0.138,+0.071,+1.53,-0.026,0.068,-0.038,-0.57,+0.018,+0.32,-0.084,0.375,+0.039,+0.100,+0.134,-0.103
Valuation|P/E chg 12M reversal,+0.080,+1.05,-0.076,-1.19,-0.218,0.863,-0.013,-0.27,-0.126,0.610,-0.128,-1.30,-0.030,-0.41,-0.207,0.641,-0.102,-0.051,+0.000,-0.082
Valuation|P/E chg 3M reversal,+0.049,+0.68,+0.029,+0.46,-0.108,0.330,+0.034,+0.73,-0.072,0.233,+0.006,+0.09,+0.025,+0.50,-0.084,0.290,+0.025,+0.033,+0.120,-0.102
Valuation|P/E percentile own 10y,+0.063,+0.80,-0.065,-0.89,-0.218,0.834,-0.010,-0.19,-0.114,0.598,-0.045,-0.52,+0.039,+0.54,-0.131,0.308,-0.063,-0.067,+0.105,-0.292


=== Technical ===


,disc3 IC,disc3 t,val3 IC,val3 t,val3 lo,val3 p,full3 IC,full3 t,full3 lo,full3 p,val6 IC,val6 t,full6 IC,full6 t,full6 lo,full6 p,val 1st half,val 2nd half,leg multiple,leg earnings
signal,,,,,,,,,,,,,,,,,,,,
"Technical|mom 1M (reversal, inv)",-0.049,-0.80,-0.047,-0.88,-0.154,0.812,-0.047,-1.20,-0.127,0.887,-0.042,-0.91,-0.033,-0.90,-0.105,0.845,+0.002,-0.096,+0.094,-0.135
Technical|mom 3M,+0.013,+0.14,+0.040,+0.63,-0.081,0.266,+0.029,+0.57,-0.060,0.297,+0.045,+0.73,+0.028,+0.58,-0.028,0.249,+0.009,+0.071,-0.134,+0.304
Technical|mom 6M,-0.003,-0.04,+0.037,+0.65,-0.067,0.254,+0.017,+0.36,-0.075,0.359,+0.080,+1.21,+0.058,+1.02,-0.044,0.158,+0.100,-0.024,-0.163,+0.338
Technical|mom 12M,+0.043,+0.47,+0.085,+1.21,-0.062,0.092,+0.058,+1.04,-0.053,0.153,+0.060,+0.68,+0.065,+0.99,-0.050,0.168,+0.180,-0.007,-0.051,+0.274
Technical|mom 12-1M,+0.038,+0.51,+0.070,+0.93,-0.089,0.161,+0.049,+0.90,-0.059,0.174,+0.049,+0.51,+0.054,+0.80,-0.066,0.225,+0.175,-0.033,-0.054,+0.255
Technical|mom 6-1M,-0.023,-0.26,+0.036,+0.51,-0.097,0.307,+0.016,+0.30,-0.092,0.386,+0.064,+0.89,+0.055,+0.89,-0.067,0.207,+0.148,-0.073,-0.143,+0.290
Technical|relative strength 6M,-0.003,-0.04,+0.037,+0.65,-0.067,0.254,+0.017,+0.36,-0.075,0.359,+0.080,+1.21,+0.058,+1.02,-0.044,0.158,+0.100,-0.024,-0.163,+0.338
Technical|price vs 6M MA,+0.008,+0.09,+0.040,+0.61,-0.097,0.274,+0.023,+0.45,-0.069,0.335,+0.053,+0.81,+0.027,+0.50,-0.040,0.289,+0.020,+0.060,-0.175,+0.333
Technical|vol-adjusted mom 12-1M,+0.010,+0.13,+0.112,+1.57,-0.035,0.042,+0.056,+1.07,-0.047,0.143,+0.107,+1.34,+0.083,+1.38,-0.027,0.080,+0.232,-0.004,-0.012,+0.255


### Sample sensitivity: v10's common sample is SHORTER than v9's, and that alone moves the results
Adding signals that need a 10-year rolling window shortens the window on which *everything* is compared. That is a confound in its own right, so the two core signals are reported on **both** samples.

In [5]:
core = {'MACRO fci chg 6M': cs_z(FCI6), 'MICRO eps rev 6M': cs_z(EPSREV6)}
v9_valid = pd.Series({t: all(z.loc[t].notna().all() for z in core.values()) for t in IDX})
V9SAMPLE = v9_valid[v9_valid].index
rows = []
for nm, sg in core.items():
    for slab, smp in (('v10 common (44 signals)', SAMPLE), ('v9-style (2 signals only)', V9SAMPLE)):
        d0 = (smp.min(), pd.Period('2016-12', 'M')); v0 = (pd.Period('2017-07', 'M'), smp.max())
        icd = win(ic_series(sg, TGT[('total', 3)], smp), d0); icv = win(ic_series(sg, TGT[('total', 3)], smp), v0)
        td, _ = hac_t(icd.values, 3); tv, _ = hac_t(icv.values, 3)
        rows.append({'signal': nm, 'sample': slab, 'start': str(smp.min()), 'end': str(smp.max()), 'n months': len(smp),
                     'disc IC': icd.mean(), 'disc t': td, 'val IC': icv.mean(), 'val t': tv})
SENS = pd.DataFrame(rows).set_index(['signal', 'sample'])
display(SENS.style.format({'disc IC': '{:+.3f}', 'val IC': '{:+.3f}', 'disc t': '{:+.2f}', 'val t': '{:+.2f}'}))
SENS.to_csv(OUT/'sample_sensitivity.csv')
log('M02', 'method', 'Results are invariant to which common sample is used',
    'compare the two core signals on the 44-signal common sample vs the 2-signal (v9-style) sample',
    'both samples, same discovery/validation split rule',
    'see SENS table - IC and t move materially between samples',
    'Sample choice is itself a source of variation. Cross-version comparisons (v9 vs v10) are NOT like-for-like.')

## 4. Challenge: why does FCI improve out of sample while EPS weakens?
**Hypothesis:** the validation window (2017-2026) contains COVID, the 2022 tightening and the 2023-24 easing — far more cross-market dispersion in financing conditions than the ZIRP discovery window. A signal with more variation to exploit should show a higher IC. If so, FCI's "improvement" is a **regime effect, not evidence of a stronger relationship**.

In [6]:
disp_fci = (FCI - FCI.shift(6)).std(axis=1); disp_eps = (EPS / EPS.shift(6) - 1).std(axis=1)
rows = []
for lab, w in (('discovery', DISC), ('validation', VALW)):
    rows.append({'window': lab, 'FCI 6M-chg cross-market dispersion': win(disp_fci, w).mean(),
                 'EPS 6M-rev cross-market dispersion': win(disp_eps, w).mean(),
                 'VIX mean': win(VIX, w).mean()})
DISP = pd.DataFrame(rows).set_index('window')
DISP.loc['ratio val/disc'] = DISP.loc['validation'] / DISP.loc['discovery']
display(DISP.style.format('{:.3f}'))
icF = ic_series(Z['Macro|fci chg 6M'], TGT[('total', 3)], SAMPLE); icE = ic_series(Z['Micro|eps rev 6M'], TGT[('total', 3)], SAMPLE)
j = pd.concat([icF.rename('icF'), icE.rename('icE'), disp_fci.rename('dF'), disp_eps.rename('dE')], axis=1).dropna()
print(f'corr(monthly FCI IC, FCI dispersion) = {j.icF.corr(j.dF):+.2f}')
print(f'corr(monthly EPS IC, EPS dispersion) = {j.icE.corr(j.dE):+.2f}')
lo_, hi_ = j.dF.quantile([1/3, 2/3])
print(f'FCI IC by own dispersion tercile: low {j[j.dF<=lo_].icF.mean():+.3f} | mid {j[(j.dF>lo_)&(j.dF<hi_)].icF.mean():+.3f} | high {j[j.dF>=hi_].icF.mean():+.3f}')
lo2, hi2 = j.dE.quantile([1/3, 2/3])
print(f'EPS IC by own dispersion tercile: low {j[j.dE<=lo2].icE.mean():+.3f} | mid {j[(j.dE>lo2)&(j.dE<hi2)].icE.mean():+.3f} | high {j[j.dE>=hi2].icE.mean():+.3f}')
log('C01', 'Macro', 'FCI improves out of sample because the validation regime has more dispersion, not because the relationship is stronger',
    'compare cross-market dispersion of FCI 6M change across windows; IC by dispersion tercile',
    f'common sample {SAMPLE.min()}..{SAMPLE.max()}',
    f"dispersion ratio val/disc = {DISP.loc['ratio val/disc','FCI 6M-chg cross-market dispersion']:.2f}; corr(IC, dispersion) = {j.icF.corr(j.dF):+.2f}",
    'see printed terciles - if IC rises monotonically with dispersion the improvement is regime-driven')

,FCI 6M-chg cross-market dispersion,EPS 6M-rev cross-market dispersion,VIX mean
window,,,
discovery,0.619,0.052,17.388
validation,0.446,0.044,19.305
ratio val/disc,0.721,0.844,1.110


corr(monthly FCI IC, FCI dispersion) = -0.03
corr(monthly EPS IC, EPS dispersion) = -0.18
FCI IC by own dispersion tercile: low +0.106 | mid +0.108 | high +0.142
EPS IC by own dispersion tercile: low +0.208 | mid +0.192 | high -0.062


### Regime conditioning: does either signal only work in particular states?

In [7]:
vix_hi = VIX > VIX.rolling(24, min_periods=12).median()
rows = []
for nm, k in (('Macro FCI 6M', 'Macro|fci chg 6M'), ('Micro EPS 6M', 'Micro|eps rev 6M')):
    ic = ic_series(Z[k], TGT[('total', 3)], SAMPLE)
    for state, m in (('high VIX', vix_hi), ('low VIX', ~vix_hi)):
        s = ic[m.reindex(ic.index).fillna(False)]
        t, _ = hac_t(s.values, 3)
        rows.append({'signal': nm, 'regime': state, 'n': len(s), 'IC': s.mean(), 'HAC t': t})
display(pd.DataFrame(rows).set_index(['signal', 'regime']).style.format({'IC': '{:+.3f}', 'HAC t': '{:+.2f}'}))

## 5. Micro: is EPS revision driven by the US, and is the US series comparable?

In [8]:
rows = []
for nm, k in (('Macro FCI 6M', 'Macro|fci chg 6M'), ('Micro EPS 6M', 'Micro|eps rev 6M')):
    base = win(ic_series(Z[k], TGT[('total', 3)], SAMPLE), VALW)
    r = {'signal': nm, 'all 5': base.mean()}
    for c in COUNTRIES:
        keep = [x for x in COUNTRIES if x != c]
        src = {'Macro FCI 6M': FCI6, 'Micro EPS 6M': EPSREV6}[nm]
        sk = cs_z(src[keep].reindex(IDX)); tk = TGT[('total', 3)][keep].sub(TGT[('total', 3)][keep].mean(axis=1), axis=0)
        r[f'ex-{c}'] = win(ic_series(sk, tk, SAMPLE), VALW).mean()
    rows.append(r)
LOO = pd.DataFrame(rows).set_index('signal')
display(LOO.style.format('{:+.3f}'))
print('\nUS EPS comparability (Monthly_EPS vs BEst EPS):')
g = EPS.pct_change()
cmp_ = pd.DataFrame({'field': {c: SPEC[c]['eps'] for c in COUNTRIES},
                     'monthly chg sd': g.std(), 'autocorr(1)': g.apply(lambda s: s.autocorr()),
                     'share declining': (g < 0).mean(), 'share |chg|<0.1%': (g.abs() < 0.001).mean(),
                     '6M rev sd': (EPS / EPS.shift(6) - 1).std()})
display(cmp_.style.format({'monthly chg sd': '{:.4f}', 'autocorr(1)': '{:+.2f}', 'share declining': '{:.0%}', 'share |chg|<0.1%': '{:.1%}', '6M rev sd': '{:.3f}'}))
us_only = EPSREV6[['US']]; print(f"US 6M-revision mean {us_only.mean().iloc[0]:+.3%} vs non-US mean {EPSREV6[[c for c in COUNTRIES if c!='US']].stack().mean():+.3%}")
log('C02', 'Micro', 'The EPS-revision signal is driven disproportionately by the US',
    'leave-one-country-out on the validation window; compare US field statistics with the four BEst series',
    f'validation {VALW[0]}..{VALW[1]}',
    f"ex-US IC {LOO.loc['Micro EPS 6M','ex-US']:+.3f} vs all-5 {LOO.loc['Micro EPS 6M','all 5']:+.3f}",
    'see table')

,all 5,ex-US,ex-Eurozone,ex-UK,ex-Switzerland,ex-Japan
signal,,,,,,
Macro FCI 6M,+0.175,+0.090,+0.164,+0.198,+0.164,+0.220
Micro EPS 6M,+0.102,+0.076,+0.124,+0.085,+0.070,+0.124



US EPS comparability (Monthly_EPS vs BEst EPS):


,field,monthly chg sd,autocorr(1),share declining,share |chg|<0.1%,6M rev sd
US,Monthly_EPS,0.0201,+0.65,20%,4.1%,0.085
Eurozone,BEst EPS,0.0225,+0.73,32%,5.7%,0.097
UK,BEst EPS,0.0290,+0.52,42%,4.1%,0.115
Switzerland,BEst EPS,0.0199,+0.29,34%,3.7%,0.065
Japan,BEst EPS,0.0463,+0.70,31%,4.9%,0.190


US 6M-revision mean +3.700% vs non-US mean +2.210%


## 6. Incremental predictive power — both directions, all pillar pairs

In [9]:
BEST = {'Macro': 'Macro|fci chg 6M', 'Micro': 'Micro|eps rev 6M'}
def comp(keys): return sum(Z[k] for k in keys) / len(keys)
def paired(base, plus, w, h=3, lags=3, L=6):
    a = win(ic_series(comp(base), TGT[('total', h)], SAMPLE), w); b = win(ic_series(comp(base + plus), TGT[('total', h)], SAMPLE), w)
    jj = pd.concat([a.rename('a'), b.rename('b')], axis=1).dropna(); d = jj.b - jj.a
    t, _ = hac_t(d.values, lags); lo, hi, p = mbb(d.values, L)
    return {'IC base': jj.a.mean(), 'IC plus': jj.b.mean(), 'ΔIC': d.mean(), 'HAC t': t, 'lo': lo, 'hi': hi, 'add-one p': p}
PAIRS = [('Macro alone -> +Micro', [BEST['Macro']], [BEST['Micro']]),
         ('Micro alone -> +Macro', [BEST['Micro']], [BEST['Macro']])]
rows = []
for nm, b, p_ in PAIRS:
    for lab, w in (('discovery', DISC), ('validation', VALW)):
        rows.append({'step': nm, 'window': lab, **paired(b, p_, w)})
display(pd.DataFrame(rows).set_index(['step', 'window']).style.format({'IC base': '{:+.3f}', 'IC plus': '{:+.3f}', 'ΔIC': '{:+.3f}', 'HAC t': '{:+.2f}', 'lo': '{:+.3f}', 'hi': '{:+.3f}', 'add-one p': '{:.3f}'}))

## 7. Technical: is the price series contaminating the signals with earnings information?
Three separate questions, often conflated:

1. **Is $P=(P/E)\times EPS$ a fake price?** No. It is an accounting identity and v4 §12b verified the recovered US series against the **actual S&P 500** (mean absolute difference 0.04 pp across eight months, vol 15.2 %, 2008-09 drawdown −52.6 %). We are using genuine index observations.
2. **Do price signals contain earnings information?** Yes — and so do signals built on *any* real index, because real prices embed earnings. This is economics, not a data artifact.
3. **Does technical add anything beyond EPS revisions?** This is the only question that matters, and it is measured below.

**Contamination metric:** pooled $R^2$ from regressing each technical signal on the earnings block (EPS revisions 3M/6M/12M and EPS growth 12M). High $R^2$ = the signal is largely an earnings restatement.

In [10]:
EBLOCK = pd.DataFrame({'r3': cs_z(EPS / EPS.shift(3) - 1).stack(), 'r6': cs_z(EPSREV6).stack(),
                       'r12': cs_z(EPS / EPS.shift(12) - 1).stack(), 'g12': cs_z(EPSG12).stack()})
rows = []
for k in [x for x in Z if x.startswith('Technical|')]:
    y = Z[k].stack().rename('y')
    jj = pd.concat([y, EBLOCK], axis=1).dropna()
    if len(jj) < 100: continue
    X_ = sm.add_constant(jj[['r3', 'r6', 'r12', 'g12']].values)
    r2 = sm.OLS(jj['y'].values, X_).fit().rsquared
    resid = pd.Series(sm.OLS(jj['y'].values, X_).fit().resid, index=jj.index).unstack()
    resid = cs_z(resid.reindex(IDX)[COUNTRIES])
    ic_raw = win(ic_series(Z[k], TGT[('total', 3)], SAMPLE), VALW)
    ic_res = win(ic_series(resid, TGT[('total', 3)], SAMPLE), VALW)
    t_res, _ = hac_t(ic_res.values, 3)
    rows.append({'technical signal': k.split('|')[1], 'R2 on earnings block': r2, 'raw val IC': ic_raw.mean(),
                 'earnings-free val IC': ic_res.mean(), 'earnings-free HAC t': t_res,
                 'IC retained': ic_res.mean() / ic_raw.mean() if ic_raw.mean() != 0 else np.nan})
CONT = pd.DataFrame(rows).set_index('technical signal').sort_values('R2 on earnings block')
display(CONT.style.format({'R2 on earnings block': '{:.1%}', 'raw val IC': '{:+.3f}', 'earnings-free val IC': '{:+.3f}',
                           'earnings-free HAC t': '{:+.2f}', 'IC retained': '{:.0%}'}))
CONT.to_csv(OUT/'technical_contamination.csv')
print('PATH signals (vol, drawdown, trend R2, semideviation) cannot be written as a function of EPS levels;')
print('momentum ratios can, since log(P_t/P_s) = log(PE_t/PE_s) + log(EPS_t/EPS_s).')
log('C03', 'Technical', 'The reconstructed price contaminates technical signals with earnings information',
    'pooled R2 of each technical signal on the earnings block; re-test the earnings-free residual',
    f'validation {VALW[0]}..{VALW[1]}',
    f"momentum R2 range {CONT['R2 on earnings block'].max():.0%} max; path-signal R2 {CONT['R2 on earnings block'].min():.0%} min",
    'see table: price is genuine (v4), but momentum ratios do embed earnings; path signals do not')

,R2 on earnings block,raw val IC,earnings-free val IC,earnings-free HAC t,IC retained
technical signal,,,,,
"mom 1M (reversal, inv)",1.0%,-0.047,-0.043,-0.80,90%
PATH low realised vol 12M,1.5%,-0.084,-0.087,-1.34,103%
PATH trend quality R2 12M,2.3%,+0.171,+0.119,+1.88,70%
PATH small drawdown 12M,3.0%,-0.019,-0.040,-0.62,212%
PATH downside semidev 12M,5.2%,-0.027,-0.084,-1.16,313%
mom 3M,8.6%,+0.040,+0.063,+0.92,156%
price vs 6M MA,11.4%,+0.040,+0.069,+1.01,169%
mom 6M,19.7%,+0.037,+0.027,+0.45,73%
relative strength 6M,19.7%,+0.037,+0.027,+0.45,73%


PATH signals (vol, drawdown, trend R2, semideviation) cannot be written as a function of EPS levels;
momentum ratios can, since log(P_t/P_s) = log(PE_t/PE_s) + log(EPS_t/EPS_s).


## 8. Reopened pillars — selection on DISCOVERY ONLY, then one validation test
To avoid reusing the validation window for selection, the best Valuation and Technical candidates are chosen **purely on discovery-window evidence**, then tested once on validation. This is the discipline v8 violated.

In [11]:
def pick_on_discovery(pillar):
    pool = [k for k in Z if k.startswith(pillar + '|')]
    sc = {}
    for k in pool:
        ic = win(ic_series(Z[k], TGT[('total', 3)], SAMPLE), DISC)
        t, _ = hac_t(ic.values, 3); sc[k] = (ic.mean(), t)
    best = max(sc, key=lambda k: sc[k][0])
    return best, pd.DataFrame([{'candidate': k.split('|')[1], 'disc IC': v[0], 'disc HAC t': v[1]} for k, v in sc.items()]).set_index('candidate').sort_values('disc IC', ascending=False)
PICKS = {}
for p_ in ('Valuation', 'Technical'):
    best, tab = pick_on_discovery(p_); PICKS[p_] = best
    print(f'=== {p_}: discovery-only ranking (selection evidence) ==='); display(tab.style.format({'disc IC': '{:+.3f}', 'disc HAC t': '{:+.2f}'}))
    print(f'  -> selected on discovery: {best.split("|")[1]}')
rows = []
for p_, k in PICKS.items():
    ic = win(ic_series(Z[k], TGT[('total', 3)], SAMPLE), VALW)
    t, _ = hac_t(ic.values, 3); lo, hi, pv = mbb(ic.values, 6)
    rows.append({'pillar': p_, 'signal': k.split('|')[1], 'disc IC': win(ic_series(Z[k], TGT[('total', 3)], SAMPLE), DISC).mean(),
                 'VALIDATION IC': ic.mean(), 'HAC t': t, 'boot 2.5%': lo, 'boot 97.5%': hi, 'add-one p': pv})
HONEST = pd.DataFrame(rows).set_index('pillar')
print('\nONE validation test of the discovery-selected candidates (not reused for selection):')
display(HONEST.style.format({'disc IC': '{:+.3f}', 'VALIDATION IC': '{:+.3f}', 'HAC t': '{:+.2f}', 'boot 2.5%': '{:+.3f}', 'boot 97.5%': '{:+.3f}', 'add-one p': '{:.3f}'}))

=== Valuation: discovery-only ranking (selection evidence) ===


,disc IC,disc HAC t
candidate,,
P/E vs peers (cheap+),+0.136,+1.24
ERP vs credit (E/P-2Y-OAS),+0.123,+1.23
P/E z own history (cheap+),+0.110,+1.41
P/E chg 12M reversal,+0.080,+1.05
ERP z (E/P - 2Y),+0.072,+0.77
P/E percentile own 10y,+0.063,+0.80
P/E chg 6M reversal,+0.057,+0.84
P/E chg 3M reversal,+0.049,+0.68
cheap AND low vol (gate),+0.044,+0.67


  -> selected on discovery: P/E vs peers (cheap+)


=== Technical: discovery-only ranking (selection evidence) ===


,disc IC,disc HAC t
candidate,,
PATH low realised vol 12M,+0.090,+1.28
PATH small drawdown 12M,+0.057,+0.81
PATH downside semidev 12M,+0.048,+0.61
mom 12M,+0.043,+0.47
mom 12-1M,+0.038,+0.51
mom 3M,+0.013,+0.14
vol-adjusted mom 12-1M,+0.010,+0.13
price vs 6M MA,+0.008,+0.09
mom 6M,-0.003,-0.04


  -> selected on discovery: PATH low realised vol 12M



ONE validation test of the discovery-selected candidates (not reused for selection):


,signal,disc IC,VALIDATION IC,HAC t,boot 2.5%,boot 97.5%,add-one p
pillar,,,,,,,
Valuation,P/E vs peers (cheap+),+0.136,-0.064,-0.90,-0.231,+0.040,0.820
Technical,PATH low realised vol 12M,+0.090,-0.084,-1.22,-0.230,+0.031,0.887


### Do the reopened pillars add anything to Macro + Micro?

In [12]:
rows = []
for p_, k in PICKS.items():
    for lab, w in (('discovery', DISC), ('validation', VALW)):
        rows.append({'step': f'Macro+Micro -> +{p_} ({k.split("|")[1]})', 'window': lab,
                     **paired([BEST['Macro'], BEST['Micro']], [k], w)})
INC2 = pd.DataFrame(rows).set_index(['step', 'window'])
display(INC2.style.format({'IC base': '{:+.3f}', 'IC plus': '{:+.3f}', 'ΔIC': '{:+.3f}', 'HAC t': '{:+.2f}', 'lo': '{:+.3f}', 'hi': '{:+.3f}', 'add-one p': '{:.3f}'}))
INC2.to_csv(OUT/'incremental_reopened.csv'); HONEST.to_csv(OUT/'discovery_selected_validation_test.csv')

## 9. Why does cheapness predict multiple expansion but not total return?
Measured directly: the IC of the best valuation construction against each leg, and the correlation between the two legs themselves.

In [13]:
rows = []
for k in [x for x in Z if x.startswith('Valuation|')][:6]:
    r = {'valuation signal': k.split('|')[1]}
    for lab, w in (('disc', DISC), ('val', VALW)):
        r[f'{lab} multiple'] = win(ic_series(Z[k], TGT[('multiple', 3)], SAMPLE), w).mean()
        r[f'{lab} earnings'] = win(ic_series(Z[k], TGT[('earnings', 3)], SAMPLE), w).mean()
        r[f'{lab} total'] = win(ic_series(Z[k], TGT[('total', 3)], SAMPLE), w).mean()
    rows.append(r)
VLEG = pd.DataFrame(rows).set_index('valuation signal')
display(VLEG.style.format('{:+.3f}'))
m_ = TGT[('multiple', 3)].stack(); e_ = TGT[('earnings', 3)].stack()
jj = pd.concat([m_.rename('mult'), e_.rename('earn')], axis=1).dropna()
print(f'Correlation between the two legs of the SAME market-month: {jj.mult.corr(jj.earn):+.3f}')
print(f'Variance share: multiple {np.cov(jj.mult, jj.mult + jj.earn)[0,1]/np.var(jj.mult+jj.earn):.2f}, earnings {np.cov(jj.earn, jj.mult + jj.earn)[0,1]/np.var(jj.mult+jj.earn):.2f}')
hz = []
for h in (1, 3, 6, 12):
    for k in ('Valuation|P/E z own history (cheap+)', 'Valuation|P/E chg 6M reversal'):
        hz.append({'signal': k.split('|')[1], 'horizon': h,
                   'disc IC': win(ic_series(Z[k], TGT[('total', h)], SAMPLE), DISC).mean(),
                   'val IC': win(ic_series(Z[k], TGT[('total', h)], SAMPLE), VALW).mean()})
HZ = pd.DataFrame(hz).pivot(index='horizon', columns='signal', values=['disc IC', 'val IC'])
display(HZ.style.format('{:+.3f}'))
VLEG.to_csv(OUT/'valuation_leg_analysis.csv')
log('C04', 'Valuation', 'Cheapness predicts multiple expansion but the earnings leg cancels it',
    'IC of each valuation construction against the multiple, earnings and total legs; leg correlation; horizon profile',
    f'common sample, disc + val',
    f"leg correlation {jj.mult.corr(jj.earn):+.3f}; see VLEG table",
    'see table')

,disc multiple,disc earnings,disc total,val multiple,val earnings,val total
valuation signal,,,,,,
P/E z own history (cheap+),+0.162,+0.084,+0.110,+0.130,-0.173,-0.090
P/E vs peers (cheap+),+0.074,+0.174,+0.136,+0.138,-0.234,-0.064
earnings yield z,+0.069,+0.121,+0.041,+0.144,-0.192,-0.072
ERP z (E/P - 2Y),+0.170,+0.179,+0.072,+0.071,-0.153,-0.029
ERP vs credit (E/P-2Y-OAS),+0.284,-0.151,+0.123,+0.091,-0.055,+0.052
P/E chg 6M reversal,+0.111,+0.067,+0.057,+0.134,-0.103,+0.070


Correlation between the two legs of the SAME market-month: -0.767
Variance share: multiple 0.79, earnings 0.21


## 10. Cumulative multiplicity accounting
Explicit count of every selection decision made on this dataset since v1. Any p-value below is **conditional on the v10 null only** and the true burden is larger.

In [14]:
HIST = pd.DataFrame([
 {'phase': 'v1', 'what was searched': '1 pre-specified 4-pillar spec', 'candidates': 1},
 {'phase': 'v2', 'what was searched': '4 specifications (FCI swap x weighting)', 'candidates': 4},
 {'phase': 'v3', 'what was searched': '40 single signals + 3 composites', 'candidates': 43},
 {'phase': 'v4', 'what was searched': 'exhaustive 1/2/3-signal + 4-pillar combinations', 'candidates': 25301},
 {'phase': 'v5', 'what was searched': '4 repricing experiments', 'candidates': 4},
 {'phase': 'v6', 'what was searched': '21 features x 4 model families', 'candidates': 84},
 {'phase': 'v7', 'what was searched': 'verification only (no new selection)', 'candidates': 0},
 {'phase': 'v8', 'what was searched': '12 pillar candidates', 'candidates': 12},
 {'phase': 'v9', 'what was searched': '5 signals revalidated', 'candidates': 5},
 {'phase': 'v10', 'what was searched': f'{len(Z)} signals across 4 pillars', 'candidates': len(Z)},
]).set_index('phase')
HIST['cumulative'] = HIST['candidates'].cumsum()
display(HIST)
print(f'Cumulative selection decisions on one dataset: {HIST["cumulative"].iloc[-1]:,}')
print('A Bonferroni-style correction at this scale would require p < 1e-5 for any single claim.')
print('That is unattainable with 5 markets and ~180 months, which is itself the central finding of this project.')
HIST.to_csv(OUT/'multiplicity_history.csv')

,what was searched,candidates,cumulative
phase,,,
v1,1 pre-specified 4-pillar spec,1,1
v2,4 specifications (FCI swap x weighting),4,5
v3,40 single signals + 3 composites,43,48
v4,exhaustive 1/2/3-signal + 4-pillar combinations,25301,25349
v5,4 repricing experiments,4,25353
v6,21 features x 4 model families,84,25437
v7,verification only (no new selection),0,25437
v8,12 pillar candidates,12,25449
v9,5 signals revalidated,5,25454


Cumulative selection decisions on one dataset: 25,498
A Bonferroni-style correction at this scale would require p < 1e-5 for any single claim.
That is unattainable with 5 markets and ~180 months, which is itself the central finding of this project.


### v10 null with add-one correction (conditional)

In [15]:
names = list(Z)
A = np.stack([Z[k].to_numpy(float) for k in names])
maskv = np.array([t in set(SAMPLE) and VALW[0] <= t <= VALW[1] for t in IDX])
Yt = TGT[('total', 3)].to_numpy(float)
def cr(M):
    out = np.full(M.shape, np.nan); g = ~np.isnan(M).any(-1); out[g] = rankdata(M[g], axis=-1) - 3.0; return out
SR, FR = cr(A), cr(Yt)
SRz, FRz = np.nan_to_num(SR), np.nan_to_num(FR)
den = np.sqrt((SRz**2).sum(-1) * (FRz**2).sum(-1)[None, :]); den[den == 0] = np.nan
VM = (~np.isnan(SR).any(-1)) & (~np.isnan(FR).any(-1))[None, :] & ~np.isnan(den) & maskv[None, :]
mean_ic = lambda S_: np.nanmean(np.where(VM, np.einsum('itc,tc->it', S_, FRz) / den, np.nan), axis=1)
obs = mean_ic(SRz)
rng = np.random.default_rng(SEED); blk = np.arange(len(IDX)) // 3
nulls = np.empty((NPERM, len(names)))
for r in range(NPERM):
    perm = np.argsort(rng.random((blk.max() + 1, 5)), axis=1)[blk]
    nulls[r] = mean_ic(np.take_along_axis(SRz, perm[None, :, :], axis=2))
nmax = nulls.max(axis=1)
NT = pd.DataFrame({'validation IC': obs,
                   'p own null': [(1 + int((nulls[:, i] >= obs[i]).sum())) / (1 + NPERM) for i in range(len(names))],
                   f'p vs best-of-{len(names)} null': [(1 + int((nmax >= obs[i]).sum())) / (1 + NPERM) for i in range(len(names))]}, index=names)
NT['pillar'] = [n.split('|')[0] for n in NT.index]
display(NT.sort_values('validation IC', ascending=False).head(14).style.format({'validation IC': '{:+.3f}', 'p own null': '{:.4f}', f'p vs best-of-{len(names)} null': '{:.4f}'}))
print(f'Best-of-{len(names)} noise ceiling (95th pct): {np.percentile(nmax, 95):+.3f}; observed best: {obs.max():+.3f}')
NT.to_csv(OUT/'null_tests_v10.csv')

,validation IC,p own null,p vs best-of-44 null,pillar
Macro|fci chg 6M,+0.175,0.0025,0.1129,Macro
Technical|PATH trend quality R2 12M,+0.171,0.0070,0.1304,Technical
Micro|eps rev 6M (own-sd scaled),+0.161,0.0140,0.1849,Micro
Macro|fci chg 6M (rank),+0.157,0.0050,0.2179,Macro
Macro|fci chg 12M,+0.152,0.0155,0.2629,Macro
Micro|eps stability 12M (inv vol),+0.145,0.0230,0.3228,Micro
Macro|fci level z,+0.126,0.0380,0.5067,Macro
Technical|vol-adjusted mom 12-1M,+0.112,0.0415,0.6417,Technical
Micro|eps rev 6M lagged 1M,+0.108,0.0650,0.6902,Micro
Micro|eps rev 6M,+0.102,0.0745,0.7376,Micro


Best-of-44 noise ceiling (95th pct): +0.193; observed best: +0.175


## 11. Two separate views
**(1) Diagnostic scorecard** — best defensible indicator per pillar with its validation status, for reporting the four pillars the brief requires.
**(2) Predictive model** — only signals with incremental, reasonably robust evidence.

In [16]:
DIAG = {'Macro': BEST['Macro'], 'Micro': BEST['Micro'], 'Valuation': PICKS['Valuation'], 'Technical': PICKS['Technical']}
rows = []
for p_, k in DIAG.items():
    s = SWEEP.loc[k]
    inc_src = INC2 if p_ in ('Valuation', 'Technical') else None
    rows.append({'pillar': p_, 'indicator': k.split('|')[1], 'disc IC': s['disc3 IC'], 'val IC': s['val3 IC'],
                 'val HAC t': s['val3 t'], 'val boot 2.5%': s['val3 lo'], 'full IC': s['full3 IC'], 'full boot p': s['full3 p'],
                 f'p vs best-of-{len(names)}': NT.loc[k, f'p vs best-of-{len(names)} null']})
DIAGT = pd.DataFrame(rows).set_index('pillar')
display(DIAGT.style.format({c: '{:+.3f}' for c in ('disc IC', 'val IC', 'val boot 2.5%', 'full IC')} | {'val HAC t': '{:+.2f}', 'full boot p': '{:.3f}', f'p vs best-of-{len(names)}': '{:.3f}'}))
last_ok = max(t for t in SAMPLE)
cur = pd.DataFrame({p_: Z[k].loc[last_ok] for p_, k in DIAG.items()})
cur['diagnostic avg'] = cur.mean(axis=1)
cur = cur.sort_values('diagnostic avg', ascending=False)
print(f'\nCountry readings at {last_ok} (last month with all signals valid; panel ends {AS_OF}) - DIAGNOSTIC view, not an allocation:')
display(cur.style.format('{:+.2f}'))
DIAGT.to_csv(OUT/'diagnostic_scorecard.csv'); cur.reset_index(names='country').to_csv(OUT/'diagnostic_current_readings.csv', index=False)

,indicator,disc IC,val IC,val HAC t,val boot 2.5%,full IC,full boot p,p vs best-of-44
pillar,,,,,,,,
Macro,fci chg 6M,+0.048,+0.175,+2.49,+0.029,+0.119,0.015,0.113
Micro,eps rev 6M,+0.133,+0.102,+1.21,-0.082,+0.113,0.041,0.738
Valuation,P/E vs peers (cheap+),+0.136,-0.064,-0.90,-0.231,+0.027,0.345,1.000
Technical,PATH low realised vol 12M,+0.090,-0.084,-1.22,-0.230,-0.027,0.695,1.000



Country readings at 2026-04 (last month with all signals valid; panel ends 2026-04) - DIAGNOSTIC view, not an allocation:


,Macro,Micro,Valuation,Technical,diagnostic avg
US,+1.09,+1.27,-1.51,+0.16,+0.25
UK,-1.59,+0.41,+1.29,+0.86,+0.24
Eurozone,+0.32,-0.39,-0.07,+0.45,+0.08
Switzerland,+0.38,-1.43,+0.09,+0.26,-0.18
Japan,-0.20,+0.15,+0.20,-1.72,-0.39


In [17]:
def retained(k, pillar):
    s = SWEEP.loc[k]
    if pillar in ('Valuation', 'Technical'):
        r = INC2.loc[(f'Macro+Micro -> +{pillar} ({k.split("|")[1]})', 'validation')]
        adds = bool(r['ΔIC'] > 0 and r['lo'] > 0)
    else:
        adds = None
    return bool(s['val3 t'] >= 1 and s['val3 IC'] > 0), bool(s['full3 lo'] > 0), adds
rows = []
for p_, k in DIAG.items():
    val_ok, full_ok, adds = retained(k, p_)
    if p_ in ('Valuation', 'Technical'):
        status = 'RETAINED' if (val_ok and adds) else ('PROVISIONAL' if val_ok else 'REJECTED')
    else:
        status = 'PROVISIONAL' if val_ok else 'REJECTED'
    rows.append({'pillar': p_, 'indicator': k.split('|')[1], 'val IC>0 & t>=1': val_ok,
                 'full boot CI excl 0': full_ok, 'adds to Macro+Micro': adds, 'status': status})
STATUS = pd.DataFrame(rows).set_index('pillar')
display(STATUS)
STATUS.to_csv(OUT/'final_status.csv')
PRED = [k for p_, k in DIAG.items() if STATUS.loc[p_, 'status'] in ('RETAINED', 'PROVISIONAL')]
print(f'\nPREDICTIVE MODEL contains: {[k.split("|")[1] for k in PRED]}')

,indicator,val IC>0 & t>=1,full boot CI excl 0,adds to Macro+Micro,status
pillar,,,,,
Macro,fci chg 6M,True,False,None,PROVISIONAL
Micro,eps rev 6M,True,False,None,PROVISIONAL
Valuation,P/E vs peers (cheap+),False,False,False,REJECTED
Technical,PATH low realised vol 12M,False,False,False,REJECTED



PREDICTIVE MODEL contains: ['fci chg 6M', 'eps rev 6M']


## 12. Research log and exports

In [18]:
RL = pd.DataFrame(LOG)
display(RL)
RL.to_csv(OUT/'research_log.csv', index=False)
rows = []
for k in Z:
    s = SWEEP.loc[k]
    rows.append({'pillar': k.split('|')[0], 'signal': k.split('|')[1], 'disc IC 3M': s['disc3 IC'], 'val IC 3M': s['val3 IC'],
                 'val t 3M': s['val3 t'], 'val boot lo': s['val3 lo'], 'full IC 3M': s['full3 IC'], 'full boot p': s['full3 p'],
                 'val IC 6M': s['val6 IC'], 'val t 6M': s['val6 t'], 'val 1st half': s['val 1st half'], 'val 2nd half': s['val 2nd half'],
                 'leg multiple': s['leg multiple'], 'leg earnings': s['leg earnings'],
                 'p vs best-of-N null': NT.loc[k, f'p vs best-of-{len(Z)} null']})
ROB = pd.DataFrame(rows).set_index(['pillar', 'signal']).sort_index()
ROB.to_csv(OUT/'robustness_comparison_all_pillars.csv')
print('Robustness comparison table (all four pillars) written to output/v10/robustness_comparison_all_pillars.csv')
display(ROB.style.format({c: '{:+.3f}' for c in ROB.columns if 'IC' in c or 'leg' in c or 'half' in c or c.endswith('lo')} |
                         {c: '{:+.2f}' for c in ROB.columns if c.endswith(' 3M') and ' t ' in c} |
                         {'val t 3M': '{:+.2f}', 'val t 6M': '{:+.2f}', 'full boot p': '{:.3f}', 'p vs best-of-N null': '{:.3f}'}))
json.dump({'head_at_start': '81057b2', 'common_sample': [str(SAMPLE.min()), str(SAMPLE.max())], 'n_months': int(len(SAMPLE)),
           'discovery': [str(DISC[0]), str(DISC[1])], 'validation': [str(VALW[0]), str(VALW[1])],
           'n_signals_v10': len(Z), 'cumulative_candidates_all_phases': int(HIST['cumulative'].iloc[-1]),
           'inference': 'HAC lags = horizon; moving-block bootstrap L = 2 x horizon; add-one MC p-values',
           'selection_discipline': 'Valuation and Technical selected on DISCOVERY ONLY, then tested once on validation'},
          open(OUT/'v10_run_spec.json', 'w'), indent=2)
print('Saved:', *[p.name for p in sorted(OUT.glob('*.csv'))], 'v10_run_spec.json', sep='\n  ')

,id,pillar,hypothesis,construction,sample,result,conclusion
0,M01,method,The six offsets are six independent robustness...,compare mean-of-offsets with full-series mean;...,"full common sample, 6M horizon",mean-of-offsets +0.1742 = full-series mean +0....,REFUTED. Offsets are a start-date sensitivity ...
1,M02,method,Results are invariant to which common sample i...,compare the two core signals on the 44-signal ...,"both samples, same discovery/validation split ...",see SENS table - IC and t move materially betw...,Sample choice is itself a source of variation....
2,C01,Macro,FCI improves out of sample because the validat...,compare cross-market dispersion of FCI 6M chan...,common sample 2010-11..2026-04,"dispersion ratio val/disc = 0.72; corr(IC, dis...",see printed terciles - if IC rises monotonical...
3,C02,Micro,The EPS-revision signal is driven disproportio...,leave-one-country-out on the validation window...,validation 2017-07..2026-04,ex-US IC +0.076 vs all-5 +0.102,see table
4,C03,Technical,The reconstructed price contaminates technical...,pooled R2 of each technical signal on the earn...,validation 2017-07..2026-04,momentum R2 range 35% max; path-signal R2 1% min,"see table: price is genuine (v4), but momentum..."
5,C04,Valuation,Cheapness predicts multiple expansion but the ...,IC of each valuation construction against the ...,"common sample, disc + val",leg correlation -0.767; see VLEG table,see table


Robustness comparison table (all four pillars) written to output/v10/robustness_comparison_all_pillars.csv


Saved:
  diagnostic_current_readings.csv
  diagnostic_scorecard.csv
  discovery_selected_validation_test.csv
  final_status.csv
  incremental_reopened.csv
  multiplicity_history.csv
  null_tests_v10.csv
  research_log.csv
  robustness_comparison_all_pillars.csv
  sample_sensitivity.csv
  standalone_sweep.csv
  technical_contamination.csv
  valuation_leg_analysis.csv
  v10_run_spec.json


## 13. Which previous conclusions changed, and why

In [19]:
CHG = pd.DataFrame([
 {'Conclusion': 'v8/v9: "k of 6 non-overlapping offsets positive" is robustness evidence',
  'Now': 'WITHDRAWN', 'Why': 'The six offsets partition one series; their mean equals the full-series mean exactly (+0.1742 vs +0.1742). They measure start-date sensitivity, not replication. Every robustness claim resting on offsets is weaker than stated.'},
 {'Conclusion': 'v9: "discovery >> validation decay is the signature of overfitting"',
  'Now': 'PARTLY WITHDRAWN', 'Why': 'On a common sample three of five signals improved out of sample. But v10 shows the common sample itself moves results (Macro disc IC 0.048 vs 0.079 across two samples), so neither version is a clean read.'},
 {'Conclusion': 'v9: FCI improves out of sample because of a regime effect',
  'Now': 'REFUTED', 'Why': 'Validation has LOWER FCI cross-market dispersion (ratio 0.72), not higher, and corr(IC, dispersion) = -0.03. The improvement is not a dispersion artifact. Mild VIX-regime dependence remains (high-VIX IC +0.154 vs low-VIX +0.098).'},
 {'Conclusion': 'v9: the Micro EPS pillar is "the strongest standalone signal"',
  'Now': 'CHANGED', 'Why': 'On the v10 sample Macro FCI has the stronger validation evidence (IC +0.175, t 2.49, full boot p 0.015) while Micro is +0.102, t 1.21. The ranking is sample-dependent, which is itself the finding.'},
 {'Conclusion': 'v8: valuation V3 (multiple reversal) is the pillar winner',
  'Now': 'REJECTED', 'Why': 'Under discovery-only selection the best valuation candidate is P/E vs peers; it posts validation IC -0.064 and a significantly NEGATIVE incremental contribution (-0.091, HAC t -2.35, CI [-0.170,-0.025]).'},
 {'Conclusion': 'v9: technical is rejected because the price series embeds earnings',
  'Now': 'REFINED', 'Why': 'The price is genuine (v4 identity check). Momentum ratios do carry earnings information (R2 8.6-35.3% on the earnings block) but the new PATH family does not (R2 1.0-5.2%). Technical is rejected on its OUT-OF-SAMPLE RESULT, not on contamination.'},
 {'Conclusion': 'Open question: why does cheapness predict multiple expansion but not total return',
  'Now': 'ANSWERED', 'Why': 'The two legs of the same market-month correlate -0.767. Predicting one leg positively means predicting the other negatively; they offset. The multiple leg is 79% of return variance but cannot be harvested alone.'},
]).set_index('Conclusion')
display(CHG); CHG.to_csv(OUT/'changed_conclusions.csv')

,Now,Why
Conclusion,,
"v8/v9: ""k of 6 non-overlapping offsets positive"" is robustness evidence",WITHDRAWN,The six offsets partition one series; their me...
"v9: ""discovery >> validation decay is the signature of overfitting""",PARTLY WITHDRAWN,On a common sample three of five signals impro...
v9: FCI improves out of sample because of a regime effect,REFUTED,Validation has LOWER FCI cross-market dispersi...
"v9: the Micro EPS pillar is ""the strongest standalone signal""",CHANGED,On the v10 sample Macro FCI has the stronger v...
v8: valuation V3 (multiple reversal) is the pillar winner,REJECTED,Under discovery-only selection the best valuat...
v9: technical is rejected because the price series embeds earnings,REFINED,The price is genuine (v4 identity check). Mome...
Open question: why does cheapness predict multiple expansion but not total return,ANSWERED,The two legs of the same market-month correlat...


In [20]:
REC = pd.DataFrame([
 {'Pillar': 'Macro', 'Indicator': 'FCI 6M easing', 'Status': 'PROVISIONAL - strongest candidate',
  'Evidence': 'val IC +0.175 (HAC t 2.49), full-sample boot p 0.015, ex-US +0.090 (survives), adds +0.058 to Micro (t 1.08, not significant)',
  'Against': 'discovery IC only +0.048; p vs best-of-44 null = 0.113; sample-dependent'},
 {'Pillar': 'Micro', 'Indicator': 'EPS revision 6M', 'Status': 'PROVISIONAL',
  'Evidence': 'val IC +0.102 (t 1.21), full boot p 0.041, stable across LOO (ex-US +0.076), works in low-dispersion regimes',
  'Against': 'does not add to Macro in validation (-0.015); p vs best-of-44 = 0.738; US field definition unverified'},
 {'Pillar': 'Valuation', 'Indicator': 'P/E vs peers (best on discovery)', 'Status': 'REJECTED',
  'Evidence': 'none out of sample',
  'Against': 'val IC -0.064; incremental -0.091 (t -2.35, CI excludes 0 on the NEGATIVE side); 12 constructions tested, none survives'},
 {'Pillar': 'Valuation', 'Indicator': 'ERP vs credit (E/P-2Y-OAS)', 'Status': 'UNDER INVESTIGATION',
  'Evidence': 'only valuation construction with positive validation total IC (+0.052) and the largest discovery multiple-leg loading (+0.284)',
  'Against': 'not selected by the discovery rule; too weak to claim; would need a fresh window'},
 {'Pillar': 'Technical', 'Indicator': 'PATH low realised vol (best on discovery)', 'Status': 'REJECTED',
  'Evidence': 'none out of sample',
  'Against': 'val IC -0.084 (t -1.22); incremental -0.069'},
 {'Pillar': 'Technical', 'Indicator': 'PATH trend quality R2 12M', 'Status': 'UNDER INVESTIGATION',
  'Evidence': 'val IC +0.171, earnings-free val IC +0.119 (t 1.88), only 2.3% R2 on the earnings block - genuinely independent price information',
  'Against': 'discovery IC -0.011, so it was NOT selected; claiming it now would reuse the validation window for selection. Must be tested on fresh data.'},
]).set_index('Pillar')
display(REC); REC.to_csv(OUT/'final_recommendation.csv')
md = f"""
### Recommendation
**Predictive model: Macro FCI 6M easing only, as a provisional candidate.** Micro EPS revision 6M is retained as a second provisional signal on standalone grounds, but it does not add to Macro out of sample, so a two-signal model is not evidenced — it is a judgement call that costs little either way.

**Valuation and Technical are rejected for the predictive model.** Twelve valuation and thirteen technical constructions were tested; the discovery-selected candidate in each pillar produced a *negative* validation IC and a negative incremental contribution, significantly so for valuation.

**Two candidates remain under investigation and must not be used yet:** ERP-vs-credit (valuation) and trend-quality R² (technical). Both look interesting in validation but neither was selected on discovery, so promoting them now would be exactly the error this phase exists to prevent.

**For the four-pillar diagnostic view** the best defensible indicator per pillar is reported with its validation status attached, and valuation/technical are labelled as descriptive, not predictive.
"""
display(Markdown(md))

,Indicator,Status,Evidence,Against
Pillar,,,,
Macro,FCI 6M easing,PROVISIONAL - strongest candidate,"val IC +0.175 (HAC t 2.49), full-sample boot p...",discovery IC only +0.048; p vs best-of-44 null...
Micro,EPS revision 6M,PROVISIONAL,"val IC +0.102 (t 1.21), full boot p 0.041, sta...",does not add to Macro in validation (-0.015); ...
Valuation,P/E vs peers (best on discovery),REJECTED,none out of sample,"val IC -0.064; incremental -0.091 (t -2.35, CI..."
Valuation,ERP vs credit (E/P-2Y-OAS),UNDER INVESTIGATION,only valuation construction with positive vali...,not selected by the discovery rule; too weak t...
Technical,PATH low realised vol (best on discovery),REJECTED,none out of sample,val IC -0.084 (t -1.22); incremental -0.069
Technical,PATH trend quality R2 12M,UNDER INVESTIGATION,"val IC +0.171, earnings-free val IC +0.119 (t ...","discovery IC -0.011, so it was NOT selected; c..."



### Recommendation
**Predictive model: Macro FCI 6M easing only, as a provisional candidate.** Micro EPS revision 6M is retained as a second provisional signal on standalone grounds, but it does not add to Macro out of sample, so a two-signal model is not evidenced — it is a judgement call that costs little either way.

**Valuation and Technical are rejected for the predictive model.** Twelve valuation and thirteen technical constructions were tested; the discovery-selected candidate in each pillar produced a *negative* validation IC and a negative incremental contribution, significantly so for valuation.

**Two candidates remain under investigation and must not be used yet:** ERP-vs-credit (valuation) and trend-quality R² (technical). Both look interesting in validation but neither was selected on discovery, so promoting them now would be exactly the error this phase exists to prevent.

**For the four-pillar diagnostic view** the best defensible indicator per pillar is reported with its validation status attached, and valuation/technical are labelled as descriptive, not predictive.
